# dfthcsos: build, refit and train (commit mode)

Run it with **Save Version → Save & Run All (Commit)**. Everything written to `/kaggle/working` stays in the
version's **Output** tab, and after every step the best verified result so far is packaged as
`/kaggle/working/submission.zip`.

Inputs (right panel, **+ Add Input**): the competition, the *Verifiable Circuit Compilation Starter Kit* dataset and
the `dfthcsos-toolkit` dataset, which must hold the toolkit v4 (with `qat.py`) and the trained factorizations
`R55_B55` and `R55_B55_C70`. Settings: **GPU** accelerator, **Internet** on.

Plan (about 2 h 45 min):
1. `C70_b12`: build R55_B55_C70 at 12 bits, no training (x1.234, verified locally).
2. `B55_q12`: refit the weights of R55_B55 for its 12-bit vectors (x1.255 if it passes).
3. `R55_B52_C75`: train (55, 52, 75) from R55_B55_C70, then refit it at 12 bits (x1.276 if it passes).

In [ ]:
# Settings. EPOCH_SCALE < 1 shortens every run (only for quick tests); keep it at 1.0 for real runs.
EPOCH_SCALE = 1.0

In [ ]:
# 1. Locate the inputs and prepare the work folders
import functools, json, os, shutil, subprocess, sys, traceback, zipfile
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

def find(pattern, what):
    hits = list(INPUT.glob(pattern))
    if not hits:
        raise FileNotFoundError(f'{what} not found: add it with "+ Add Input" (right panel).')
    return hits[0]

OFFICIAL = INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation'
COMPETITION = OFFICIAL if (OFFICIAL / 'verifiers').is_dir() else \
    find('**/verifiers/dfthcsos/verify.py', 'the competition data').parents[2]
KIT_SRC = find('**/starter_kit/starter', 'the starter kit').parent
TOOLKIT_SRC = find('**/qat.py', 'the dfthcsos-toolkit dataset (v4, with qat.py)').parent
print('competition :', COMPETITION)
print('starter kit :', KIT_SRC)
print('toolkit     :', TOOLKIT_SRC)

# The toolkit is copied to a writable folder.
TOOLKIT = WORK / 'toolkit'
shutil.copytree(TOOLKIT_SRC, TOOLKIT, dirs_exist_ok=True)

# The trainer looks for the FCIDUMP and the fixed shifts in the folder above starter_kit/.
KIT_PARENT = WORK / 'kit'; KIT_PARENT.mkdir(exist_ok=True)
for link, target in [('starter_kit', KIT_SRC),
                     ('FeMoCo76o113e.fcidump', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fcidump'),
                     ('FeMoCo76o113e.fixed_shifts.npz', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz')]:
    if not (KIT_PARENT / link).exists():
        (KIT_PARENT / link).symlink_to(target)

os.environ.update({'COMPETITION_DIR': str(COMPETITION),
                   'KIT_DIR': str(KIT_PARENT / 'starter_kit'),
                   'WORK_DIR': str(WORK)})
print('ok')

In [ ]:
# 2. Check the GPU and the libraries
import importlib.util
if importlib.util.find_spec('optax') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)
import jax
print('JAX backend:', jax.default_backend())  # should be 'gpu'

In [ ]:
# 3. Load the toolkit and define the helpers
import numpy as np

sys.path.insert(0, str(TOOLKIT))
os.chdir(TOOLKIT)
import run_experiments as RX
import starter.task1_factorization.train_df_thc_qaggle as TQ
from dfthc import train_df_thc_sga as TS
from qat import qat as _qat

REF = RX.lab.ref()
SORTED_SEED = TOOLKIT / 'FeMoCo76o113e_65_60_60_sorted.npz'
FACTORIZATION_KEYS = ('U', 'W', 'H2_I', 'H1_I', 'Hsym_lower')


def log(record):
    """Print a result and append it to results.jsonl."""
    print(json.dumps(record, indent=1, default=float))
    with open(WORK / 'results.jsonl', 'a') as f:
        f.write(json.dumps(record, default=float) + '\n')


def package():
    """Write submission.zip from the best verified result so far (reference circuits for invsqrt and dqi)."""
    results = WORK / 'results.jsonl'
    rows = [json.loads(line) for line in open(results)] if results.exists() else []
    for r in rows:
        print(f"{r['name']:14s} status={str(r.get('status')):20s} brot={r.get('brot')} "
              f"improvement={r.get('improvement', float('nan')):.4f}")
    passed = [r for r in rows if r.get('status') == 'PASS']
    if not passed:
        print('No verified result yet: submission.zip not written.')
        return
    best = max(passed, key=lambda r: r['improvement'])
    sample = COMPETITION / 'sample_submission'
    with zipfile.ZipFile(WORK / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for name in ('invsqrt.kmb', 'dqi.kmb'):
            zf.write(sample / name, name)
        for path in sorted(Path(best['folder']).iterdir()):
            zf.write(path, f'dfthcsos/{path.name}')
    print(f"submission.zip <- {best['name']} (improvement x{best['improvement']:.4f})")


def seed_npz(name, search_root=INPUT, out_dir=WORK / 'seeds'):
    """Return a .npz for factorization `name`, rebuilding it if Kaggle unpacked it into a folder of .npy files."""
    files = [p for p in Path(search_root).glob(f'**/{name}.npz') if p.is_file()]
    if files:
        arrays = dict(np.load(files[0]))
    else:
        folders = [p for p in Path(search_root).glob(f'**/{name}') if (p / 'U.npy').is_file()]
        if not folders:
            raise FileNotFoundError(f'{name}: neither {name}.npz nor a folder with U.npy under {search_root}')
        arrays = {key: np.load(folders[0] / f'{key}.npy') for key in FACTORIZATION_KEYS}
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    path = Path(out_dir) / f'{name}.npz'
    np.savez(path, **{key: arrays[key] for key in FACTORIZATION_KEYS})
    print('seed:', path, arrays['U'].shape)
    return path


def build_only(name, source, brot):
    """Build and verify a trained factorization at `brot` bits (with the weight refit), log it and repackage."""
    record = {'name': name, 'source': str(source)}
    try:
        folder = WORK / 'builds' / f'{name}_b{brot}' / 'dfthcsos'
        m = RX.lab.build_folder(source, folder, brot=brot, refit=True, r=REF)
        record[f'b{brot}_local'] = {k: m[k] for k in ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff')}
        summary = RX.verify(folder)
        record.update(brot=brot, folder=str(folder), status=summary.get('status'),
                      cost=summary.get('score'), error=summary.get('error'))
        if summary.get('status') == 'PASS':
            record['improvement'] = RX.BASELINE_COST / summary['score']
    except Exception:
        traceback.print_exc()
        record['status'] = 'ERROR'
    log(record)
    package()


def run_qat(name, source, brot, epochs=20_000):
    """Freeze the vectors at their brot-bit values, retrain only the weights, build, verify and repackage."""
    try:
        _qat(name, source, RX=RX, TQ=TQ, TS=TS, REF=REF, WORK=WORK, brot=brot,
             epochs=max(20, int(epochs * EPOCH_SCALE)))
    except Exception:
        traceback.print_exc()
    package()


def train(name, seed=None, lr_final=1e-6, **cfg):
    """Train, build and verify one config; the learning rate decays exponentially from lr to lr_final."""
    cfg['epochs'] = max(20, int(cfg['epochs'] * EPOCH_SCALE))
    RX.SEED = Path(seed) if seed and Path(seed).exists() else SORTED_SEED
    print(f'=== {name}: seed {RX.SEED.name}, {cfg}')
    TQ.Train = functools.partial(TS.Train, learning_rate_final=lr_final)
    try:
        RX.run(name, cfg, REF)
    except Exception:
        traceback.print_exc()
    package()

## Seeds

In [ ]:
C70_SEED = seed_npz('R55_B55_C70')
B55_SEED = seed_npz('R55_B55')

## Step 1: R55_B55_C70 at 12 bits, no training (a few minutes; x1.234)

In [ ]:
build_only('C70_b12', C70_SEED, brot=12)

## Step 2: weight refit of R55_B55 at 12 bits (about 25 min; x1.255 if it passes)

In [ ]:
run_qat('B55_q12', B55_SEED, brot=12)

## Step 3: train (55, 52, 75) from R55_B55_C70, then refit it at 12 bits (about 2 h 15 min; x1.276 if it passes)

In [ ]:
train('R55_B52_C75', seed=C70_SEED, rank=55, bases=52, copies=75, epochs=80_000,
      lr=5e-5, lr_final=1e-6, rho=1e-4, epsilon_dfthc=0.015)
b52 = WORK / 'runs' / 'R55_B52_C75.npz'
if b52.exists():
    run_qat('B52C75_q12', b52, brot=12)

## Clean up
Keeps `submission.zip`, `results.jsonl`, the trained factorizations (`runs/`) and the builds.

In [ ]:
os.chdir(WORK)
for path in (WORK / 'toolkit', WORK / 'kit', WORK / 'femoco76_ref_cache.npz', WORK / 'seeds'):
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
print(sorted(p.name for p in WORK.iterdir()))